# 2026.10.15(목) 22h~

transformer 모델의 큰 흐름 이해하기

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers

# ==========================================
# 1. 트랜스포머 인코더 블록 정의 (레고 블록)
# ==========================================
def transformer_encoder_block(embed_dim, num_heads, ff_dim, rate=0.1):
    inputs = layers.Input(shape=(None, embed_dim)) # 처리할 토큰의 개수가 가변적
    
    # [Step 1] Multi-Head Attention (단어 간 관계 파악)
    # 전문가(Head)들이 각기 다른 관점으로 문장을 분석합니다.
    attention_output = layers.MultiHeadAttention(
        num_heads=num_heads, key_dim=embed_dim
    )(inputs, inputs)
    attention_output = layers.Dropout(rate)(attention_output) # multi head attention 결과의 일부를 랜덤하게? 꺼줌 -> 과적합 방지
    
    # Residual Connection + Layer Norm (정보 보존 및 정규화)
    out1 = layers.LayerNormalization(epsilon=1e-6)(inputs + attention_output) # multi head attention 거친 후에 기존 입력 정보 유실되는 것에 대한 방안으로써, 기존 정보 유지/보존 = 실험을 통해 얻어진 알고리즘
    
    # [Step 2] Feed Forward Network (데이터 의미 심화 학습), sequential 기반 모델에서 비선형성 도입하는 hidden layer 같은 역할, 토큰에서 비선형성 발견해서? 데이터 심화
    ffn_output = layers.Dense(ff_dim, activation="relu")(out1)
    ffn_output = layers.Dense(embed_dim)(ffn_output)
    ffn_output = layers.Dropout(rate)(ffn_output)
    
    # Residual Connection + Layer Norm
    return tf.keras.Model(inputs=inputs, outputs=layers.LayerNormalization(epsilon=1e-6)(out1 + ffn_output))

# ==========================================
# 2. 위치 정보(Position) 포함 임베딩 층
# ==========================================
class TokenAndPositionEmbedding(layers.Layer):    
    def __init__(self, maxlen, vocab_size, embed_dim):
        super().__init__()
        self.token_emb = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim) # 토큰/단어 embedding
        self.pos_emb = layers.Embedding(input_dim=maxlen, output_dim=embed_dim) # positional embedding
    
    def call(self, x):
        maxlen = tf.shape(x)[-1]
        positions = tf.range(start=0, limit=maxlen, delta=1)
        positions = self.pos_emb(positions)
        x = self.token_emb(x)
        return x + positions
    
# ==========================================
# 3. 데이터 준비 (스마트 팩토리 가상 로그 - 한글 샘플)
# ==========================================
# 실제 강의 시에는 CSV 파일을 로드하거나 아래와 같이 샘플을 만듭니다.
vocab_size = 5000  # 단어 사전 크기
maxlen = 20        # 문장 최대 길이

# 실제로는 엑셀(CSV) 파일을 불러오지만, 여기서는 실습을 위해 직접 정의
# 적절한 학습이 되기 위해서는 Big Data 필요 수만~수백만 데이터 필요
# 아래 소량의 데이터로는 학습이 전혀 안됨
# [정상 로그: 라벨 0]
normal_logs = [
    "장비 가동 상태 정상", "현재 온도 25도 유지 중", "진동 수치 안정적임",
    "정기 점검 완료 이상 없음", "모터 회전수 정상 범위", "센서 데이터 송신 중",
    "시스템 전원 온료", "주변 습도 최적화 상태", "라인 가동율 95퍼센트",
    "냉각 시스템 정상 작동"
] 
# [위험/불량 로그: 라벨 1]
danger_logs = [
    "급격한 온도 상승 감지", "모터 소음 비정상 발생", "센서 연결 끊김 위험",
    "긴급 정지 버튼 작동", "진동 수치 임계치 초과", "부품 마모 심함 교체 필요",
    "오일 누유 발생 즉시 확인", "배선 합선 우려 전원 차단 필요", "생산품 불량률 급증",
    "장비 과부하 경고 발생"
] 
# 데이터 합치기
all_texts = normal_logs + danger_logs
all_labels = [0] * len(normal_logs) + [1] * len(danger_logs)

# 텍스트를 숫자로 바꾸는 '단어 사전' 만들기 (강의 핵심!)
vectorize_layer = layers.TextVectorization(
    max_tokens=2000,
    output_mode='int',
    output_sequence_length=maxlen
)
vectorize_layer.adapt(all_texts) # 단어 공부시키기

# 문장을 숫자로 변환
X_train = vectorize_layer(all_texts)
y_train = np.array(all_labels)

print(f"학습 데이터 준비 완료: 총 {len(all_texts)}개의 로그")

# ==========================================
# 4. 전체 모델 조립 (Classification 모델)
# ==========================================
embed_dim = 32  # 단어당 표현 숫자 개수
num_heads = 2   # 어텐션 헤드 개수
ff_dim = 32     # 내부 신경망 크기

inputs = layers.Input(shape=(maxlen,))

# 1단계: 단어 의미 + 위치 정보 부여
x = TokenAndPositionEmbedding(maxlen, vocab_size, embed_dim)(inputs)

# 2단계: 트랜스포머 블록 통과 (핵심 연산)
x = transformer_encoder_block(embed_dim, num_heads, ff_dim)(x)

# 3단계: 문장 정보 압축 및 출력 (Global Average Pooling)
x = layers.GlobalAveragePooling1D()(x) # 하나의 압축된 벡터로 요약?
x = layers.Dropout(0.1)(x)
x = layers.Dense(20, activation="relu")(x)
outputs = layers.Dense(2, activation="softmax")(x) # 출력 2개 = 다항 분류

model = tf.keras.Model(inputs=inputs, outputs=outputs)

# 모델 컴파일
model.compile(optimizer="adam", 
              loss="sparse_categorical_crossentropy", # 내부적으로 one hot encoding 일어남
              metrics=["accuracy"])

# 모델 구조 출력
# (build 없이 바로 summary()를 찍어도 됩니다 - 아래 markdown 셀 참고)
#  이 모델은 Functional API로 만들었고 맨 앞에 layers.Input(shape=(maxlen,))이 있어서,
#  모델을 만드는 시점에 이미 입력 모양이 (None, 20)으로 정해져 있습니다. -> 층마다 모양/파라미터 수를 바로 계산할 수 있음
#  (1_1_10 Sequential 예제에서 model.build(input_shape=(None, 5))가 필요했던 건 맨 앞에 Input이 없었기 때문)
model.summary()

학습 데이터 준비 완료: 총 20개의 로그


Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)           │ (None, 20)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ token_and_position_embedding_2       │ (None, 20, 32)              │         160,640 │
│ (TokenAndPositionEmbedding)          │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ functional_4 (Functional)            │ (None, 20, 32)              │          10,656 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling1d_2           │ (None, 32)                  │               0 │
│ (GlobalAveragePooling1D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_11 (Dropout)                 │ (None, 32)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_10 (Dense)                     │ (None, 20)                  │             660 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_11 (Dense)                     │ (None, 2)                   │              42 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 171,998 (671.87 KB)

 Trainable params: 171,998 (671.87 KB)

 Non-trainable params: 0 (0.00 B)


--- 모델 학습 시작 ---
Epoch 1/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 19ms/step - accuracy: 0.4000 - loss: 0.7914
Epoch 2/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.5500 - loss: 0.7265
Epoch 3/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.5000 - loss: 0.7327
Epoch 4/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.6000 - loss: 0.7023
Epoch 5/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.5000 - loss: 0.6802
Epoch 6/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.6000 - loss: 0.6226
Epoch 7/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.6500 - loss: 0.6104
Epoch 8/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.7500 - loss: 0.5848
Epoch 9/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.6000 - loss: 0.6102
Epoch 10/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.7500 - loss: 0.5637
Epoch 11/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.7000 - loss: 0.6102
Epoch 12/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.650

### Q. `model.build()` 없이 바로 `model.summary()`를 찍어도 모델 모양을 볼 수 있나?

**이 노트북에서는 네, 바로 찍어도 됩니다.** `010_seq2seq/1_1_10_숫자5개_역순_Basic.ipynb`에서는 `model.build(input_shape=(None, 5))`를 먼저 했는데, 모델을 만드는 방식이 달라서예요.

**핵심: "입력 모양을 모델이 이미 아는가?"**

summary()가 각 층의 출력 모양(`Output Shape`)과 파라미터 수(`Param #`)를 보여주려면 **맨 앞 입력이 어떤 모양인지** 알아야 해요. (예: Dense의 가중치 개수 = 입력 차원 × 출력 차원 + 편향 이므로 입력 차원을 알아야 계산됨) 입력 모양을 아는 것을 Keras에서는 **모델이 "built(만들어짐)" 상태**라고 해요.

| | 이 노트북 | 1_1_10 노트북 |
|---|---|---|
| 모델 만드는 방식 | Functional API | Sequential (`model.add(...)`) |
| 맨 앞 입력 | `inputs = layers.Input(shape=(maxlen,))` **있음** | `Embedding(10, 100, input_length=5)`만 있고 `Input` 층은 없음 |
| 모델 정의 직후 입력 모양 | `(None, 20)`으로 이미 정해짐 | 아직 모름 (`input_length=5`를 써도 Keras 3에서는 deprecated라 무시되어 built=False. 실행 시 경고 "Argument `input_length` is deprecated. Just remove it."가 나옴) |
| built 상태 | **True** | **False** |
| summary()를 바로 찍으면 | 모양/파라미터 수가 다 나옴 | `Output Shape`가 `?`, `Param #`가 `0 (unbuilt)`로 나옴 |
| 필요한 것 | 없음 | `model.build(input_shape=(None, 5))` 또는 첫 `fit`/`predict`(실제 데이터가 들어올 때) |

**Functional API는 왜 알 수 있나?** 이 노트북 코드는 `inputs = layers.Input(shape=(maxlen,))`로 시작해서 `x = 층(inputs)`, `x = 층(x)` ... 순서로 **텐서를 층에 통과시키며** 연결했어요. 이렇게 연결하는 순간 Keras가 각 층의 출력 모양을 따라 계산해 두기 때문에, `tf.keras.Model(inputs, outputs)`를 만든 시점에 이미 모든 층의 모양이 확정돼요.

**직접 실행해 확인한 결과 (voice-ai 환경, Keras 3)**
- 이 노트북의 모델: 바로 `summary()` → 모든 층의 모양이 나오고 `Total params: 171,998`.
- `Input` 없는 Sequential(Embedding→BiLSTM→Dense)에 `summary()`만 호출 → 에러는 나지 않지만 모든 층이 `?` / `0 (unbuilt)`로 나오고 `model.built`는 `False`. `model.build(input_shape=(None, 5))` 후에는 `(None, 5, 100)`, `(None, 5, 128)`, `(None, 5, 10)`와 파라미터 수가 나오고 `built`가 `True`.
- `Sequential([layers.Input(shape=(5,)), ...])`처럼 **맨 앞에 `Input`을 넣으면** build 없이도 `built=True`.

**정리**
- build()는 "입력 모양을 알려주는 용도"일 뿐 학습이나 가중치 값과는 상관없어요.
- 입력 모양을 알려주는 방법은 세 가지: (1) 맨 앞에 `Input(shape=...)` 두기, (2) `model.build(input_shape=...)`, (3) 실제 데이터로 `fit`/`predict`/모델 호출.
- 참고: `model.build(input_shape=(None, 5))`의 `None`은 배치 크기(샘플 수)로, 아직 정해지지 않았다는 뜻이고 `Input(shape=(5,))`에서는 배치 축을 빼고 쓰는 것과 같은 관계예요.

**참고 - 이 노트북의 모델 요약에서 눈여겨볼 부분**: `transformer_encoder_block(...)`은 별도의 `tf.keras.Model`이라 summary에서 `functional (Functional)` 한 줄로 묶여 나와요(출력 `(None, 20, 32)`, 파라미터 10,656). 블록 안쪽 층까지 보려면 `encoder_block.summary()`처럼 블록 모델을 따로 찍으면 돼요.


### Q. `class TokenAndPositionEmbedding(layers.Layer)`는 괄호 안이 `layers.Layer`인데, `TokenAndPositionEmbedding(maxlen, vocab_size, embed_dim)(inputs)`는 왜 모양이 다른가?

**결론: 같은 괄호 `( )`처럼 보이지만 세 군데의 괄호는 서로 다른 일을 해요. 파이썬 기본 문법이고, 아래처럼 나눠서 보면 돼요.**

```python
class TokenAndPositionEmbedding(layers.Layer):          # ① 클래스를 "정의"할 때의 괄호
    def __init__(self, maxlen, vocab_size, embed_dim):  # ② 객체를 만들 때 받는 인자
        ...
    def call(self, x):                                   # ③ 객체를 "호출"할 때 받는 인자
        ...

TokenAndPositionEmbedding(maxlen, vocab_size, embed_dim)(inputs)
#  └─────────────── ② 객체 생성 ───────────────────────┘└─ ③ 호출 ─┘
```

**① 클래스 정의의 `(layers.Layer)` = 상속(부모 클래스)**
- `class A(B):` 는 "A는 B를 물려받는다(상속)"는 뜻이고, 인자를 넘기는 게 아니에요.
- 즉 `TokenAndPositionEmbedding`은 Keras의 `layers.Layer`(층의 기본 틀)를 물려받은 **새로운 종류의 층**이에요. 덕분에 모델에 층으로 끼워 넣을 수 있고, `call()` 방식으로 호출할 수 있어요.
- 같은 이유로 `layers.Dense`, `layers.Embedding`도 모두 `layers.Layer`를 상속한 클래스예요. 우리가 직접 만든 층도 같은 틀 위에 있는 거예요.

**② `TokenAndPositionEmbedding(maxlen, vocab_size, embed_dim)` = 객체(층) 생성**
- `클래스이름(값들)`은 객체를 만드는 문법이고, 이때 넘긴 값들은 `__init__(self, maxlen, vocab_size, embed_dim)`로 전달돼요. (`self`는 파이썬이 자동으로 채워주므로 넘기는 값은 3개)
- 여기서는 설정값만 받아서 안에 `Embedding` 두 개(단어용, 위치용)를 준비해 두는 단계예요. 아직 데이터는 안 들어갔어요.
- 비교: `layers.Dense(20, activation="relu")`도 똑같이 "Dense 클래스를 만들면서 설정값 20, relu를 `__init__`에 넘기는" 거예요.

**③ `(inputs)` = 만들어진 객체를 함수처럼 호출**
- Keras의 층 객체는 **함수처럼 호출할 수 있어요**(파이썬의 `__call__`). `객체(데이터)`라고 쓰면 내부에서 `call(self, x)`가 실행돼요. 이때 `inputs`가 `x`로 전달돼요.
- 그래서 `TokenAndPositionEmbedding(maxlen, vocab_size, embed_dim)(inputs)`는 한 줄로 쓴 거지만 **두 단계**예요.

```python
layer = TokenAndPositionEmbedding(maxlen, vocab_size, embed_dim)  # 1단계: 층 만들기 (② __init__)
x = layer(inputs)                                                   # 2단계: 층에 데이터 통과시키기 (③ call)
```

- 다른 층도 마찬가지: `layers.Dense(20, activation="relu")(x)` = Dense 층을 만들고 바로 x를 통과시킴. `layers.GlobalAveragePooling1D()(x)`는 설정값이 없어서 앞 괄호가 비어 있을 뿐이에요.

**왜 두 번 괄호가 나오나? (인자의 역할이 달라서)**

| 괄호 | 무엇을 넘기나 | 받는 곳 | 언제 정해지나 |
|---|---|---|---|
| 앞 괄호 `(maxlen, vocab_size, embed_dim)` | **설정값** (사전 크기, 최대 길이, 임베딩 차원) | `__init__` | 모델을 설계할 때 |
| 뒤 괄호 `(inputs)` | **통과시킬 데이터** (텐서) | `call` | 모델이 연결/실행될 때 |

**직접 실행해 확인한 결과 (voice-ai 환경)**
- `TokenAndPositionEmbedding(20, 5000, 32)`로 만든 객체의 상속 순서는 `TokenAndPositionEmbedding → keras Layer → ...` 이고 함수처럼 호출 가능(`callable`=True)이에요.
- 객체를 만든 직후에는 `built=False`, `layers.Input(shape=(20,))`로 호출하자 출력이 `(None, 20, 32)`가 되고 `built=True`가 돼요. (이 노트북에서는 `Input`을 통과시키며 연결하는 시점에 모양이 정해지는 거예요. 바로 위 `model.summary()` 질문과 같은 이야기)
- 숫자 배열 `(2, 20)`을 넣어도 `(2, 20, 32)`가 나와요.

**이 노트북에서의 흐름**
`inputs`: `(None, 20)` 정수 토큰 → `TokenAndPositionEmbedding`: `token_emb(x) + pos_emb(positions)` → `(None, 20, 32)`. 단어 의미 벡터에 위치 벡터를 더하는 부분이에요 (Positional Encoding을 학습형 Embedding으로 구현. 논문의 sin/cos 대신 학습되는 방식이고, 논문 Table 3 (E)에서 둘의 성능이 비슷했던 부분이에요. `012_transformer/Positional_Encoding_정리.md` 5.2절 참고).

**주의 (다른 경우와 혼동하기 쉬운 점)**
- `class A(B):`의 괄호에는 부모 클래스 **이름**이 들어가고(값을 넘기는 게 아님), `A(...)`의 괄호에는 `__init__`이 받을 **값**이 들어가요.
- 코드의 `super().__init__()`은 부모(`layers.Layer`)의 준비 작업을 먼저 해달라는 호출이에요. 빠뜨리면 Keras 층으로서 제대로 동작하지 않아요.


In [ ]:
# ==========================================
# 5. 학습 및 액셔너블 AI 테스트 함수
# ==========================================
print("\n--- 모델 학습 시작 ---")
model.fit(X_train, y_train, epochs=20, batch_size=8)

In [ ]:
def predict_and_action(text_list):
    # 실제 강의 시에는 여기에 TextVectorization 과정을 포함시킵니다.
    # 여기서는 구조 이해를 위해 가상 숫자로 테스트합니다.
    #sample_input = np.random.randint(1, vocab_size, (1, maxlen))
    #prediction = model.predict(sample_input)
    #danger_prob = prediction[0][1]

    # 1. 실제 텍스트를 모델이 이해할 수 있는 숫자(Vector)로 변환
    # vectorize_layer는 학습 데이터로 미리 .adapt() 되어 있어야 합니다.
    tokenized_input = vectorize_layer(text_list)
    
    # 2. 모델 예측 실행
    prediction = model.predict(tokenized_input, verbose=0)
    danger_prob = prediction[0][1]

    
    print(f"입력 : {text_list}")
    
    print(f"\n[입력 로그 분석 결과]")
    print(f"위험 감지 확률: {danger_prob*100:.2f}%")
    
    if danger_prob > 0.5:
        print("🚨 ACTION: 장비 가동 중단 및 정비팀 알림 전송!")
    else:
        print("✅ STATUS: 정상 가동 유지")

    print("\n\n\n")

# 테스트 실행
predict_and_action(["모터 진동 이상 감지"])

In [7]:
# 테스트 실행
# 안정
predict_and_action(["모터 진동 이상 감지되었지만 동작 이상 없는 수준"])
predict_and_action(["장비 안정적 유지중."])
predict_and_action(["장비 가동 상태 정상."])
# 위험
predict_and_action(["불량 증가. 센서 정보 확인 필요"])
predict_and_action(["불량 발샹. 즉시 센서 정보 확인 필요. 생산 불량 발생 가능"])
predict_and_action(["급격한 불량 증가. 즉시 센서 정보 확인 필요. 장비 계속 구동시 생산 불량 발생 가능"])


predict_and_action(["기분이 좋고 상쾌합니다"])
predict_and_action(["상황이 좋아서 크게 걱정 없습니다"])

predict_and_action(["마음이 불안하고 걱정됩니다"])
predict_and_action(["무슨일이 일어날지도 모릅니다"])

입력 : ['모터 진동 이상 감지되었지만 동작 이상 없는 수준']

[입력 로그 분석 결과]
위험 감지 확률: 40.15%
✅ STATUS: 정상 가동 유지




입력 : ['장비 안정적 유지중.']

[입력 로그 분석 결과]
위험 감지 확률: 39.98%
✅ STATUS: 정상 가동 유지




입력 : ['장비 가동 상태 정상.']

[입력 로그 분석 결과]
위험 감지 확률: 15.52%
✅ STATUS: 정상 가동 유지




입력 : ['불량 증가. 센서 정보 확인 필요']

[입력 로그 분석 결과]
위험 감지 확률: 68.20%
🚨 ACTION: 장비 가동 중단 및 정비팀 알림 전송!




입력 : ['불량 발샹. 즉시 센서 정보 확인 필요. 생산 불량 발생 가능']

[입력 로그 분석 결과]
위험 감지 확률: 88.26%
🚨 ACTION: 장비 가동 중단 및 정비팀 알림 전송!




입력 : ['급격한 불량 증가. 즉시 센서 정보 확인 필요. 장비 계속 구동시 생산 불량 발생 가능']

[입력 로그 분석 결과]
위험 감지 확률: 90.19%
🚨 ACTION: 장비 가동 중단 및 정비팀 알림 전송!




입력 : ['기분이 좋고 상쾌합니다']

[입력 로그 분석 결과]
위험 감지 확률: 42.27%
✅ STATUS: 정상 가동 유지




입력 : ['상황이 좋아서 크게 걱정 없습니다']

[입력 로그 분석 결과]
위험 감지 확률: 46.79%
✅ STATUS: 정상 가동 유지




입력 : ['마음이 불안하고 걱정됩니다']

[입력 로그 분석 결과]
위험 감지 확률: 42.27%
✅ STATUS: 정상 가동 유지




입력 : ['무슨일이 일어날지도 모릅니다']

[입력 로그 분석 결과]
위험 감지 확률: 42.27%
✅ STATUS: 정상 가동 유지




